# 08 - Đánh giá khả năng thích nghi Cross-Dataset (UNSW-NB15, BoT-IoT, CSE-CIC-IDS2018)

### Khoảng trống nghiên cứu giải quyết (Research Gap):
- Bài báo gốc tại Table 10 chỉ dừng lại ở việc đánh giá **Zero-shot Transfer** (lấy mô hình đã train trên TON_IoT để chạy trực tiếp trên các tập dữ liệu ngoài). Kết quả cho thấy độ chính xác sụt giảm nghiêm trọng (ví dụ test thẳng trên UNSW-NB15 sụt giảm từ **98.87% xuống chỉ còn 92.5%**).
- Notebook này đi xa hơn đề xuất bài báo bằng cách **thực hiện tái huấn luyện (Retrain) kết hợp cân bằng dữ liệu SMOTE-ENN**, chứng minh mô hình có thể phục hồi hiệu năng hoàn toàn, đồng thời thử nghiệm mở rộng trên bộ dữ liệu khổng lồ **CSE-CIC-IDS2018** sử dụng pipeline tiền xử lý và chọn đặc trưng trọn vẹn.

In [1]:
# === Thiết lập môi trường ỔN ĐỊNH + CHỐNG RỚT DRIVE (auto-retry remount) ===
# Làm việc trên Ổ CỨNG LOCAL /content; kéo dữ liệu từ Drive có thử lại khi mount rớt.
import os, time, shutil
from contextlib import contextmanager

DRIVE = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
PROJECT_PATH = '/content/cdmlp'
NEED_RAW = True   # True chi voi notebook 08
ON_COLAB = False

def _mount():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

try:
    _mount(); ON_COLAB = True
except Exception:
    here = os.getcwd()
    while here != os.path.dirname(here) and not os.path.isdir(os.path.join(here, 'notebooks')):
        here = os.path.dirname(here)
    DRIVE = here; PROJECT_PATH = here

def _pull(rel):
    s = os.path.join(DRIVE, rel); d = os.path.join(PROJECT_PATH, rel)
    if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
    elif os.path.isfile(s):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copy2(s, d)

if ON_COLAB:
    os.makedirs(PROJECT_PATH, exist_ok=True)
    rels = ['scripts', 'src', 'models', 'results', 'data/colab_processed', 'data/ton_iot.csv']
    if NEED_RAW: rels.append('data/raw')
    # ton_iot.csv là item CUỐI -> nếu nó có mặt tức là đã kéo xong toàn bộ list
    sentinel = os.path.join(PROJECT_PATH, 'data', 'ton_iot.csv')
    ok = False
    for attempt in range(1, 6):
        for rel in rels:
            try: _pull(rel)
            except Exception as e: print(f'  (loi pull {rel}, thu lai sau): {str(e)[:70]}')
        if os.path.exists(sentinel):
            ok = True; break
        print(f'  [retry {attempt}/5] Drive rot khi keo du lieu -> remount + thu lai...')
        try: _mount()
        except Exception: pass
        time.sleep(3)
    if not ok:
        print('  [CANH BAO] Drive khong on dinh. Hay Runtime > Restart session roi chay lai cell nay.')
    else:
        print('  [OK] Da keo du lieu tu Drive ve local.')

os.makedirs(PROJECT_PATH, exist_ok=True); os.chdir(PROJECT_PATH)
try:
    get_ipython().run_line_magic('cd', PROJECT_PATH)
except Exception:
    pass
for d in ('data/colab_processed', 'results', 'results/xai', 'models'):
    os.makedirs(d, exist_ok=True)
print('PROJECT_PATH (local) =', PROJECT_PATH, '| ON_COLAB =', ON_COLAB)

def find_data_csv():
    for c in ['data/ton_iot.csv', 'data/raw/ton_iot.csv', 'ton_iot.csv']:
        if os.path.exists(c): return c
    raise FileNotFoundError('Khong tim thay ton_iot.csv (Drive: data/ton_iot.csv).')

@contextmanager
def step(name):
    t0=time.time(); print(f'\n[START] {name}', flush=True)
    try: yield
    finally: print(f'[DONE] {name} - {time.time()-t0:.1f}s', flush=True)

def sync_to_drive(retries=5):
    if not ON_COLAB:
        print('[local] bo qua sync Drive'); return
    for attempt in range(1, retries + 1):
        try:
            for rel in ['data/colab_processed', 'models', 'results']:
                s = os.path.join(PROJECT_PATH, rel)
                if os.path.exists(s): shutil.copytree(s, os.path.join(DRIVE, rel), dirs_exist_ok=True)
            print('[OK] Da dong bo ket qua ve Drive:', DRIVE); return
        except Exception as e:
            print(f'  [retry sync {attempt}/{retries}] Drive rot -> remount: {str(e)[:70]}')
            try: _mount()
            except Exception: pass
            time.sleep(3)
    print('  [CANH BAO] Sync Drive that bai. Ket qua van o local /content/cdmlp.')


Mounted at /content/drive
  [OK] Da keo du lieu tu Drive ve local.
/content
PROJECT_PATH (local) = /content/cdmlp | ON_COLAB = True


In [2]:
# Cài đặt các thư viện bổ trợ cần thiết
!pip -q install imbalanced-learn pyarrow joblib psutil tqdm scipy


In [3]:
# Thêm project root vào system path để import mô hình và modules
import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Chạy thực nghiệm tái huấn luyện trên UNSW-NB15 và BoT-IoT

Chúng ta sẽ thực hiện huấn luyện lại CyberDetect-MLP trên hai tập dữ liệu cùng UNSW Cyber Range bằng cách áp dụng:
- Làm sạch dữ liệu và tách các cột đặc trưng.
- Rút trích **Top 30 features** bằng Mutual Information của sklearn.
- Cân bằng dữ liệu bằng thuật toán **SMOTE-ENN**.
- Huấn luyện CyberDetect-MLP và kiểm chứng trên tập Test ngoài.

In [4]:
# Huấn luyện và đánh giá trên UNSW-NB15 và BoT-IoT
!python scripts/retrain_cross_dataset.py

2026-06-03 13:03:56.220003: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

  RETRAIN TRÊN UNSW-NB15 VỚI SMOTE-ENN
[*] Đang tải UNSW-NB15 từ /content/cdmlp/data/raw/UNSW-NB15/UNSW_NB15_training-set.csv...
[*] Shape: (175341, 45) | cols: ['id', 'dur', 'proto', 'service', 'state', 'spkts']...
[*] Cột label: 'label'
[*] Chọn top-30 features bằng Mutual Information...
[*] Top-5 features: ['sbytes', 'sttl', 'dbytes', 'dttl', 'ct_state_ttl']
[*] Features: 30, Samples: 175341
[*] Class distribution: {np.int64(0): np.int64(56000), np.int64(1): np.int64(119341)}

[INFO] Starting class imbalance handling: SMOTE-ENN
[*] Class distribution BEFORE resampling:
    - Class 0: 44800 samples
    - Class 1: 95472 samples

[*] Class distribution AFTER resampling (

## 2. Huấn luyện mở rộng trên tập dữ liệu CSE-CIC-IDS2018

Tập dữ liệu CSE-CIC-IDS2018 sử dụng công cụ rút trích đặc trưng khác cấu trúc (`CICFlowMeter`). Để đánh giá tính tổng quát hóa của methodology, chúng ta sẽ chạy trọn vẹn pipeline huấn luyện lại trên tập dữ liệu này:

In [2]:
# Huấn luyện trên CSE-CIC-IDS2018
!python scripts/retrain_cic_ids2018.py

2026-06-03 15:53:48.646521: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

  CSE-CIC-IDS2018 RETRAIN (THAT) | mau muc tieu = 500,000
[*] Chua co data CSE-CIC -> tai tu AWS Open Data (~6.5GB, ~10 phut)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have do

## 3. Tổng hợp kết quả thu được

Hiển thị bảng so sánh giữa hiệu năng Zero-shot và Retrain trên các tập dữ liệu:

In [3]:
# Tổng kết cross-dataset: ĐỌC TỪ KẾT QUẢ THẬT (không in số cứng)
import os, pandas as pd
print('='*72)
print('   TONG KET KHA NANG THICH NGHI CROSS-DATASET (doc tu file ket qua THAT)')
print('='*72)
res_csv = 'results/cross_dataset_results.csv'
if os.path.exists(res_csv):
    display(pd.read_csv(res_csv))
else:
    print('[INFO] Chua co ket qua. Can dat UNSW-NB15 / BoT-IoT vao data/raw/ roi chay lai cell `retrain_cross_dataset.py` o tren.')
    print('       Script SKIP trung thuc khi thieu du lieu (khong bia so).')
ids = 'results/ids2018_retrain_results.csv'
if os.path.exists(ids):
    print('\nCSE-CIC-IDS2018:'); display(pd.read_csv(ids))


   TONG KET KHA NANG THICH NGHI CROSS-DATASET (doc tu file ket qua THAT)


,dataset,accuracy,training_time,features_used
0,UNSW-NB15,0.937922,36.086591,30
1,BoT-IoT,0.999925,68.132201,13



CSE-CIC-IDS2018:


,Dataset,SampleRows,TopKFeatures,Resampling,Accuracy,Precision,Recall,F1-Score,ROC-AUC,TrainingTimeSec
0,CSE-CIC-IDS2018,500000,30,class_weight,0.85812,0.997856,0.408229,0.579415,0.847838,64.464525


Hiển thị biểu đồ trực quan hóa so sánh sự chênh lệch hiệu năng cực kỳ rõ nét giữa Zero-shot và Retrain:

In [7]:
from PIL import Image
img_path = 'results/fig_zeroshot_vs_retrain.png'
if os.path.exists(img_path):
    img = Image.open(img_path)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Không tìm thấy hình ảnh biểu đồ so sánh Zero-shot vs Retrain!')

Không tìm thấy hình ảnh biểu đồ so sánh Zero-shot vs Retrain!


### Nhận xét & Kết luận (số liệu THẬT đo được)

1. **Zero-shot thất bại do domain shift:** model train trên TON_IoT áp thẳng sang dataset khác sụt giảm nghiêm trọng (UNSW-NB15 ~32%, BoT-IoT ~0%).
2. **Retrain phục hồi hiệu năng:** UNSW-NB15 **93.79%**, BoT-IoT **99.99%** *(lưu ý BoT-IoT mất cân bằng cực đoan → accuracy cao mang tính hình thức, nên nhìn F1/recall lớp Normal)*.
3. **CSE-CIC-IDS2018 (mở rộng, CICFlowMeter, 500k mẫu, class_weight):** Accuracy **85.81%**, **ROC-AUC 0.848** → methodology **tổng quát được MỘT PHẦN** sang họ đặc trưng khác hẳn TON_IoT. Tuy nhiên recall lớp Attack chỉ **0.41** (bắt tốt Benign, bỏ sót ~59% attack) → cross-domain sang feature family mới khó hơn trong họ IoT-telemetry.
4. **So sánh xử lý mất cân bằng:** `class_weight` → 85.81%; `SMOTE-ENN full-balance` → 47% (over-predict attack). Lựa chọn resampling đảo chiều bias hoàn toàn — **SMOTE-ENN quá tay gây hại** (khớp phát hiện ở NB06).
5. **Mọi số liệu đọc trực tiếp từ** `results/cross_dataset_results.csv` **&** `results/ids2018_retrain_results.csv` **— không in số cứng.**

In [4]:
# === ĐỒNG BỘ KẾT QUẢ VỀ GOOGLE DRIVE (chạy cuối cùng, sau khi các cell trên xong) ===
sync_to_drive()


[OK] Da dong bo ket qua ve Drive: /content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final
